# **HW6 – VQE, Quantum Kernels, and QSVMs**
_Time required: ~2–3 hours (for students with Qiskit variational algorithm experience)_

**What you’ll practice**
- Building molecular Hamiltonians and ansatz circuits for VQE
- Hybrid quantum-classical optimization in VQE
- Brief exploration of SQD/SKQD concepts
- Constructing quantum feature maps and kernels
- Training and evaluating QSVMs
- Comparing quantum vs classical kernels
- Applications in quantum chemistry and ML classification

**What to turn in**
- This single notebook (`HW6_YourName.ipynb`) with **all cells run**, code and short written answers filled in where prompted.

**Rules & hints**
- Use **Qiskit** (version ~1.0 or later), **Qiskit Nature** for VQE, **Qiskit Machine Learning** for QSVM.
- Use AerSimulator for reproducibility.
- If stuck, explain reasoning; partial credit for clear work.
- For VQE, use small molecules like H2; for QSVM, use toy datasets like moons.
- Install qiskit-nature and qiskit-machine-learning if needed.


In [ ]:
# --- Setup (run me first) ---
# Install if needed (uncomment in Colab)
# !pip install qiskit qiskit-aer qiskit-ibm-runtime qiskit-nature qiskit-machine-learning matplotlib sklearn

# Import necessary modules
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.algorithms import VQE
from qiskit_nature.second_q.circuit.library import UCCSD
from qiskit_algorithms.optimizers import COBYLA
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC
from qiskit.circuit.library import ZZFeatureMap
from sklearn import datasets, svm, metrics
import numpy as np
import matplotlib.pyplot as plt

# Simulator backend
sim = AerSimulator()

# Function to run circuit and get counts
def get_counts(circ, shots=2000):
    tc = transpile(circ, sim)
    result = sim.run(tc, shots=shots).result()
    return result.get_counts()

def assert_close(A, B, tol=1e-4):
    if not np.allclose(A, B, atol=tol):
        raise AssertionError(f"Not close: {A} vs {B}")


## Part A — Molecular Hamiltonians & Ansatz in VQE (≈25 min)

**A1.** Use PySCFDriver to build Hamiltonian for H2 at 0.735 Å. Map to qubits with Jordan-Wigner.  
**A2.** Build UCCSD ansatz for H2 (2 qubits). Draw circuit.  
**A3.** Short answer: Explain second quantization to Pauli mapping; why UCCSD chemically inspired.


In [ ]:
# A1. H2 Hamiltonian
driver = PySCFDriver(atom="H 0 0 0; H 0 0 0.735", basis="sto-3g")
problem = driver.run()
hamiltonian = problem.hamiltonian.second_q_op()
mapper = JordanWignerMapper()
qubit_op = mapper.map(hamiltonian)
print("Qubit Hamiltonian: ", qubit_op)

# A2. UCCSD ansatz
ansatz = UCCSD(num_qubits=2, num_particles=problem.num_particles, num_spin_orbitals=problem.num_spin_orbitals)
ansatz.draw('mpl')
plt.show()

# A3. Written answer: (Fermion ops → Pauli strings via JW; UCCSD from unitary coupled-cluster, conserves particle number/spin)


## Part B — VQE Workflow (≈30 min)

**B1.** Set up VQE for H2 with COBYLA optimizer (maxiter=100). Run on simulator.  
**B2.** Print computed ground state energy; compare to exact (-1.136 Ha).  
**B3.** Short answer: Variational principle bound; role of SPSA vs COBYLA in noisy settings.


In [ ]:
# B1. VQE setup
optimizer = COBYLA(maxiter=100)
vqe = VQE(ansatz=ansatz, optimizer=optimizer, quantum_instance=sim)
result = vqe.compute_minimum_eigenvalue(operator=qubit_op)

# B2. Energy
energy = result.eigenvalue.real
print("Computed energy: ", energy)
exact_energy = -1.136  # Approx value
assert_close(energy, exact_energy, tol=0.01)

# B3. Written answer: (<H> ≥ E0 always; SPSA stochastic for noise robustness, COBYLA gradient-free but smoother)


## Part C — SQD/SKQD & Advanced VQE (≈20 min)

**C1.** Short answer: Explain SQD concept (stochastic dynamics in VQE).  
**C2.** Compare SKQD vs standard VQE (conceptual; no code).  
**C3.** Written reflection: How SQD/SKQD might improve QML under noise.


In [ ]:
# C1. Written answer: (Stochastic evolution via Kraus maps in VQE loop; models noise probabilistically)

# C2. Written answer: (SKQD adds kernel density estimation for smoother landscapes; better convergence in noisy regimes)

# C3. Written answer: (Robust to shot noise/variational barren plateaus; potential for quantum autoencoders/feature extraction)


## Part D — Quantum Kernels & Feature Maps (≈25 min)

**D1.** Build ZZFeatureMap for 2 features, reps=2. Draw circuit.  
**D2.** Compute kernel matrix for two points x1=[0.5,0.5], x2=[-0.5,-0.5] using FidelityQuantumKernel.  
**D3.** Short answer: Why Hilbert space dimension 4 for 2 qubits; kernel positive semi-definite.


In [ ]:
# D1. ZZFeatureMap
feature_map = ZZFeatureMap(feature_dimension=2, reps=2)
feature_map.draw('mpl')
plt.show()

# D2. Kernel matrix
kernel = FidelityQuantumKernel(feature_map=feature_map)
x1 = [0.5, 0.5]
x2 = [-0.5, -0.5]
K = kernel.construct_kernel_matrix(np.array([x1, x2]))
print("Kernel matrix:\n", K)

# D3. Written answer: (2^2=4 basis states; K= <phi(x)|phi(x')>^2 >=0, Hermitian by construction)


## Part E — QSVM Classification (≈30 min)

**E1.** Generate moons dataset (n_samples=100, noise=0.1). Train QSVM with ZZ map.  
**E2.** Train classical RBF SVM; compare accuracies on test set.  
**E3.** Short answer: Potential quantum advantage; noise impact on kernels.


In [ ]:
# E1. QSVM on moons
X, y = datasets.make_moons(n_samples=100, noise=0.1)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
qsvc = QSVC(quantum_kernel=kernel)
qsvc.fit(X_train, y_train)
acc_q = metrics.accuracy_score(y_test, qsvc.predict(X_test))
print("QSVM acc: ", acc_q)

# E2. Classical RBF
svc = svm.SVC(kernel='rbf')
svc.fit(X_train, y_train)
acc_c = metrics.accuracy_score(y_test, svc.predict(X_test))
print("RBF acc: ", acc_c)

# E3. Written answer: (Complex data where quantum embeddings separate better; noise reduces effective dim, washes out advantage)
